In [1]:
pip install transformers datasets evaluate scikit-learn nltk


     -------------------------------------- 515.2/515.2 kB 1.9 MB/s eta 0:00:00
     ---------------------------------------- 84.1/84.1 kB ? eta 0:00:00
     ---------------------------------------- 8.1/8.1 MB 10.8 MB/s eta 0:00:00
     ---------------------------------------- 1.5/1.5 MB 10.7 MB/s eta 0:00:00
     --------------------------------------- 27.5/27.5 MB 11.1 MB/s eta 0:00:00
     -------------------------------------- 119.7/119.7 kB 6.8 MB/s eta 0:00:00
     -------------------------------------- 144.5/144.5 kB 8.4 MB/s eta 0:00:00
     ------------------------------------- 201.0/201.0 kB 12.7 MB/s eta 0:00:00
     ---------------------------------------- 36.4/36.4 MB 9.8 MB/s eta 0:00:00
     ------------------------------------- 309.1/309.1 kB 18.7 MB/s eta 0:00:00
     -------------------------------------- 457.7/457.7 kB 9.5 MB/s eta 0:00:00
     ---------------------------------------- 67.6/67.6 kB ? eta 0:00:00
     ---------------------------------------- 44.1/44.1 


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -q kaggle



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import json, os

data = {
    "username": "raipratyush",
    "key": "KGAT_df05d04423f75199ab66d8eb3a929a6e"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(data, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("kaggle.json created!")


kaggle.json created!


In [8]:
!kaggle datasets download -d ankurzing/sentiment-analysis-for-financial-news
!unzip sentiment-analysis-for-financial-news.zip


Dataset URL: https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news
License(s): CC-BY-NC-SA-4.0
sentiment-analysis-for-financial-news.zip: Skipping, found more recently modified local copy (use --force to force download)


'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [11]:
import pandas as pd

rows = []

with open("FinancialPhraseBank/Sentences_AllAgree.txt", encoding="latin1") as f:
    for line in f:
        line = line.strip()
        if "@" in line:
            text, label = line.rsplit("@", 1)   # split from right side
            rows.append([text, label])

df = pd.DataFrame(rows, columns=["text", "label"])

print(df.head())
print(df["label"].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: 'FinancialPhraseBank/Sentences_AllAgree.txt'

In [6]:
label_map = {"negative":0, "neutral":1, "positive":2}
df["label"] = df["label"].map(label_map)

print(df.head())
print(df["label"].value_counts())


                                                text  label
0  According to Gran , the company has no plans t...      1
1  For the last quarter of 2010 , Componenta 's n...      2
2  In the third quarter of 2010 , net sales incre...      2
3  Operating profit rose to EUR 13.1 mn from EUR ...      2
4  Operating profit totalled EUR 21.1 mn , up fro...      2
label
1    1391
2     570
0     303
Name: count, dtype: int64


In [7]:
df.to_csv("dataset.csv", index=False)
print("Saved dataset.csv with", len(df), "rows")


Saved dataset.csv with 2264 rows


In [8]:
# =========================
# 1. Load dataset
# =========================
import pandas as pd

df = pd.read_csv("dataset.csv")

print("Dataset size:", len(df))
print(df.head())
print("Label distribution:")
print(df["label"].value_counts())

# =========================
# 2. Train / Test split
# =========================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# =========================
# 3. Build TF-IDF + Logistic model
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_df=0.95,
        min_df=2,
        stop_words="english"
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# =========================
# 4. Train
# =========================
print("Training model...")
model.fit(X_train, y_train)

# =========================
# 5. Evaluate
# =========================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["negative","neutral","positive"]))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# =========================
# 6. Save model
# =========================
import joblib

joblib.dump(model, "tfidf_sentiment_model.joblib")
print("\nModel saved as tfidf_sentiment_model.joblib")

# =========================
# 7. Test on custom sentences
# =========================
label_map_rev = {0:"negative", 1:"neutral", 2:"positive"}

def predict(text):
    pred = model.predict([text])[0]
    return label_map_rev[pred]

print("\nManual tests:\n")
tests = [
    "This amendment increases compliance burden and should be removed",
    "This clause is acceptable and well written",
    "The reporting procedure is unchanged"
]

for t in tests:
    print(t, "=>", predict(t))


Dataset size: 2264
                                                text  label
0  According to Gran , the company has no plans t...      1
1  For the last quarter of 2010 , Componenta 's n...      2
2  In the third quarter of 2010 , net sales incre...      2
3  Operating profit rose to EUR 13.1 mn from EUR ...      2
4  Operating profit totalled EUR 21.1 mn , up fro...      2
Label distribution:
label
1    1391
2     570
0     303
Name: count, dtype: int64
Train size: 1811
Test size: 453
Training model...

Accuracy: 0.8631346578366446

Classification Report:

              precision    recall  f1-score   support

    negative       0.71      0.82      0.76        61
     neutral       0.93      0.92      0.93       278
    positive       0.78      0.75      0.76       114

    accuracy                           0.86       453
   macro avg       0.81      0.83      0.82       453
weighted avg       0.87      0.86      0.86       453


Confusion Matrix:

[[ 50   2   9]
 [  7 256  15]
 [ 

In [9]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
# ==========================================
# POLICY SENTIMENT TRAINING PIPELINE (ADVANCED)
# ==========================================

import pandas as pd
import numpy as np
import torch
import nltk
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

nltk.download("punkt")

# ==========================================
# 1. LOAD DATA
# ==========================================
df = pd.read_csv("dataset.csv")
df = df.dropna()
print("Dataset size:", len(df))
print(df["label"].value_counts())

# ==========================================
# 2. SENTENCE-LEVEL SPLITTING (IMPORTANT)
# ==========================================
rows = []
for _, row in df.iterrows():
    sentences = nltk.sent_tokenize(row["text"])
    for s in sentences:
        if len(s.strip()) > 10:
            rows.append({"text": s, "label": row["label"]})

df = pd.DataFrame(rows)
print("After sentence splitting:", len(df))

# ==========================================
# 3. TRAIN TEST SPLIT
# ==========================================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

# ==========================================
# 4. TOKENIZER & MODEL
# ==========================================
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# ==========================================
# 5. TOKENIZATION
# ==========================================
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

# ==========================================
# 6. CLASS WEIGHTS (CRITICAL)
# ==========================================
labels = train_df["label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

# ==========================================
# 7. CUSTOM TRAINER WITH WEIGHTED LOSS
# ==========================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==========================================
# 8. METRICS
# ==========================================
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "precision_macro": precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall_macro": recall.compute(predictions=preds, references=labels, average="macro")["recall"],
    }

# ==========================================
# 9. TRAINING ARGS
# ==========================================
args = TrainingArguments(
    output_dir="./policy_sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)

# ==========================================
# 10. TRAIN
# ==========================================
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

# ==========================================
# 11. FINAL EVALUATION
# ==========================================
print("\nFINAL EVALUATION:")
preds = trainer.predict(test_ds)

y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

# ==========================================
# 12. SAVE MODEL
# ==========================================
trainer.save_model("./policy_sentiment_final")
tokenizer.save_pretrained("./policy_sentiment_final")

print("\nModel saved to ./policy_sentiment_final")

# ==========================================
# 13. CONFIDENCE TEST
# ==========================================
from transformers import pipeline

clf = pipeline("text-classification", model="./policy_sentiment_final", tokenizer=tokenizer, return_all_scores=True)

tests = [
    "This amendment increases compliance burden and should be removed",
    "This clause is acceptable and well written",
    "The reporting procedure is unchanged"
]

for t in tests:
    out = clf(t)[0]
    print("\nTEXT:", t)
    for o in out:
        print(o)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Dataset size: 2264
label
1    1391
2     570
0     303
Name: count, dtype: int64
After sentence splitting: 2278


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

              precision    recall  f1-score   support

    negative       0.98      1.00      0.99        61
     neutral       1.00      0.99      0.99       281
    positive       0.97      0.99      0.98       114

    accuracy                           0.99       456
   macro avg       0.99      0.99      0.99       456
weighted avg       0.99      0.99      0.99       456



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./policy_sentiment_final


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


TEXT: This amendment increases compliance burden and should be removed
label
score

TEXT: This clause is acceptable and well written
label
score

TEXT: The reporting procedure is unchanged
label
score


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
!cp -r policy_sentiment_final /content/drive/MyDrive/
